# Save Vector Embedding V1
Run this notebook in Kaggle to encode chunk texts from `chunk_store.jsonl` using the dual-encoder checkpoint and save embeddings to `/kaggle/working/vector_embedding_v1`.

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
DEFAULT_KAGGLE_CHUNK_STORE_PATH = Path(
    "/kaggle/input/datasets/trinhthaison/chunk-store-legal-answer-question-v1/chunk_store.jsonl"
)
DEFAULT_KAGGLE_CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/trinhthaison/dual-encoder-resume-v1-legal-question-answer-v1/best_dual_encoder_resume_v1.pt"
)
DEFAULT_OUTPUT_DIR = Path("/kaggle/working/vector_embedding_v1")

def get_jsonl_data(file_path: Path) -> List[Dict]:
    if not file_path.exists():
        raise FileNotFoundError(f"JSONL file not found: {file_path}")

    rows: List[Dict] = []
    with file_path.open("r", encoding="utf-8") as f:
        for line_no, raw_line in enumerate(f, start=1):
            line = raw_line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                preview = line[:120]
                raise ValueError(
                    f"Invalid JSON at {file_path}:{line_no}. Preview: {preview!r}. Error: {e}"
                ) from e

    if not rows:
        raise ValueError(f"No valid JSON records found in: {file_path}")
    return rows

def find_single_file(input_root: Path, filename: str) -> Path:
    matches = sorted(input_root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {input_root}")
    if len(matches) > 1:
        print(f"Found multiple {filename}; using first match: {matches[0]}")
    return matches[0]

class DualEncoder(torch.nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_name)
        self.d_encoder = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    def encode_docs(self, d_tok: Dict[str, torch.Tensor]) -> torch.Tensor:
        out = self.d_encoder(
            input_ids=d_tok["input_ids"],
            attention_mask=d_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, d_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

def apply_lora_if_needed(model: DualEncoder, cfg: Dict) -> DualEncoder:
    lora_r = int(cfg.get("lora_r", 0) or 0)
    if lora_r <= 0:
        return model

    try:
        from peft import LoraConfig, TaskType, get_peft_model
    except Exception as e:
        raise ImportError(
            "Checkpoint expects LoRA adapters but `peft` is not available. Install dependencies first."
        ) from e

    lora_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=lora_r,
        lora_alpha=int(cfg.get("lora_alpha", 32)),
        lora_dropout=float(cfg.get("lora_dropout", 0.1)),
        target_modules=cfg.get("lora_target_modules", ["query", "key", "value"]),
        bias="none",
    )
    model.q_encoder = get_peft_model(model.q_encoder, lora_cfg)
    model.d_encoder = get_peft_model(model.d_encoder, lora_cfg)
    return model

def load_checkpoint_compat(path: Path) -> Dict:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def normalize_whitespace(text: str) -> str:
    return " ".join((text or "").split())

def collect_chunk_rows(chunk_store_path: Path) -> Tuple[List[Dict], List[str]]:
    rows = get_jsonl_data(chunk_store_path)
    kept_rows: List[Dict] = []
    texts: List[str] = []

    for row in rows:
        text = normalize_whitespace(row.get("text", ""))
        if not text:
            continue
        kept_rows.append(
            {
                "doc_id": row.get("doc_id"),
                "article_id": row.get("article_id"),
                "chunk_id": row.get("chunk_id"),
                "text": text,
            }
        )
        texts.append(text)

    if not texts:
        raise ValueError("No non-empty `text` entries found in chunk_store.jsonl")

    return kept_rows, texts

def batched_encode_docs(
    model: DualEncoder,
    tokenizer,
    texts: List[str],
    max_len: int,
    device: torch.device,
    batch_size: int,
    progress_every_batches: int,
) -> torch.Tensor:
    all_emb = []
    total_texts = len(texts)
    total_batches = (total_texts + batch_size - 1) // batch_size if total_texts > 0 else 0

    if total_batches > 0:
        print(f"Start encoding {total_texts} chunks in {total_batches} batches...")

    with torch.no_grad():
        for batch_idx, i in enumerate(range(0, len(texts), batch_size), start=1):
            batch_texts = texts[i : i + batch_size]
            d_tok = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            d_tok = {k: v.to(device) for k, v in d_tok.items()}
            d_emb = model.encode_docs(d_tok)
            all_emb.append(d_emb.cpu())

            should_print = (
                batch_idx == 1
                or batch_idx == total_batches
                or (progress_every_batches > 0 and batch_idx % progress_every_batches == 0)
            )
            if should_print:
                processed = min(batch_idx * batch_size, total_texts)
                pct = (processed / total_texts) * 100 if total_texts else 100.0
                print(
                    f"[Encode Progress] batches={batch_idx}/{total_batches} | "
                    f"chunks={processed}/{total_texts} ({pct:.2f}%)"
                )

    return torch.cat(all_emb, dim=0) if all_emb else torch.empty((0, 1), dtype=torch.float32)

def save_outputs(output_dir: Path, embeddings: torch.Tensor, chunk_rows: List[Dict]) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    emb_path = output_dir / "vector_embeddings_v1.pt"
    meta_path = output_dir / "chunk_metadata_v1.jsonl"

    torch.save(embeddings, emb_path)
    with meta_path.open("w", encoding="utf-8") as f:
        for row in chunk_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Saved embeddings to: {emb_path}")
    print(f"Saved metadata to:   {meta_path}")
    print(f"Embedding shape:     {tuple(embeddings.shape)}")

In [ ]:
# Runtime config (edit these if needed)
chunk_store_path = DEFAULT_KAGGLE_CHUNK_STORE_PATH
checkpoint_path = DEFAULT_KAGGLE_CHECKPOINT_PATH
output_dir = DEFAULT_OUTPUT_DIR
batch_size = 64
progress_every_batches = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if IN_KAGGLE:
    if not chunk_store_path.exists():
        print(f"Given chunk_store_path does not exist: {chunk_store_path}. Trying auto-discovery...")
        chunk_store_path = find_single_file(Path("/kaggle/input"), "chunk_store.jsonl")
    if not checkpoint_path.exists():
        print(f"Given checkpoint_path does not exist: {checkpoint_path}. Trying auto-discovery...")
        checkpoint_path = find_single_file(Path("/kaggle/input"), "best_dual_encoder_resume_v1.pt")

print(f"IN_KAGGLE={IN_KAGGLE}")
print(f"Using chunk_store_path: {chunk_store_path}")
print(f"Using checkpoint_path:  {checkpoint_path}")
print(f"Using output_dir:       {output_dir}")

chunk_rows, chunk_texts = collect_chunk_rows(chunk_store_path)
print(f"Loaded {len(chunk_texts)} chunk texts from: {chunk_store_path}")

checkpoint = load_checkpoint_compat(checkpoint_path)
cfg = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
model_name = checkpoint.get("model_name") if isinstance(checkpoint, dict) else None
model_name = model_name or cfg.get("model_name") or "vinai/phobert-base"
max_c_len = int(cfg.get("max_c_len", 128))

print(f"Model: {model_name} | max_c_len={max_c_len} | device={device}")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = DualEncoder(model_name)
model = apply_lora_if_needed(model, cfg)

state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys while loading: {len(missing)}")
if unexpected:
    print(f"Unexpected keys while loading: {len(unexpected)}")

model = model.to(device)
model.eval()

embeddings = batched_encode_docs(
    model=model,
    tokenizer=tokenizer,
    texts=chunk_texts,
    max_len=max_c_len,
    device=device,
    batch_size=batch_size,
    progress_every_batches=progress_every_batches,
)

save_outputs(output_dir=output_dir, embeddings=embeddings, chunk_rows=chunk_rows)